# Lesson 02 — From raw files to clean data tables

**Course:** Python from zero for Materials Science PhD students  
**Session:** Hands-on 2  
**Nominal duration:** 3 hours  
**Realistic target for the first group:** about 2 hours of core material + optional exercises

## Aim of this lesson

In the first hands-on session, we worked mostly with values written directly in the notebook.

Today we move one step closer to real scientific work:

```text
file on disk → table in Python → inspected data → calculated columns → clean output file
```

We will use a small synthetic tensile-test dataset.

The goal is not to become experts in pandas today. The goal is to become comfortable with the basic operations that appear in almost every data-analysis workflow:

- finding a data file;
- reading a CSV file;
- inspecting rows and columns;
- selecting columns;
- filtering rows;
- creating new calculated columns;
- checking simple data-quality issues;
- saving cleaned data and summary tables.

## Important idea

A data-analysis notebook should be repeatable.

This means that if we run the notebook from top to bottom, it should produce the same cleaned data and the same results.

## 0. Setup

We will use a few standard libraries:

- `pathlib`, from the Python standard library, to handle file paths;
- `pandas`, to read and manipulate tables;
- `matplotlib`, to make a very simple check plot at the end;
- `math`, for constants such as `math.pi` (we met this in the previous lesson).

Today, `pandas` is the main tool.

Conventionally, pandas is imported as `pd`.

In [ ]:
from pathlib import Path
import math

import pandas as pd
import matplotlib.pyplot as plt

print("Setup OK")

## 1. Files and paths

A file path tells Python where a file is located.

For this lesson, the data files are expected to be in a folder called `data`, so the main file is:

```text
data/tensile_lab_raw.csv
```

A CSV file is a plain-text table where values are separated by commas.

We build paths with `pathlib.Path` and join folders and file names with the `/` operator. This works the same way on every operating system.

> **Note.** On a normal computer you can check whether a file exists with `raw_file.exists()`. Inside the browser (JupyterLite) that check is unreliable, so instead of asking *"does the file exist?"* we simply **try to read the file and handle the failure**. Trying-and-handling is a good habit for input/output in general.

In [ ]:
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")

# Create the results folder if it does not exist yet
RESULTS_DIR.mkdir(exist_ok=True)

# Build the path to the main data file ("data/tensile_lab_raw.csv")
raw_file = DATA_DIR / "tensile_lab_raw.csv"

print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Raw data file:", raw_file)

### Locating the file robustly

The data might not be exactly where we expect:

1. the `data` folder was not uploaded;
2. the CSV file was uploaded next to the notebook, not inside `data`;
3. the notebook was opened from a different working folder.

To handle all of these cases, we define a small helper, `load_table()`, that **tries to read the file from several locations** and uses the first one that works:

1. the `data` folder;
2. the current folder;
3. as a last resort, it downloads the file directly from the course website.

We will reuse this helper every time we read a CSV in this lesson.

In [ ]:
def load_table(name):
    """Read a CSV by name, trying several locations (browser-safe).

    Order: data/ -> current folder -> download from the served site.
    Returns a pandas DataFrame.
    """
    # 1) and 2): try local folders first
    for folder in ("data", ".", ".."):
        try:
            return pd.read_csv(Path(folder) / name)
        except Exception:
            continue
    # 3) last resort: fetch the file over HTTP from the deployed site
    import js
    from pyodide.http import open_url
    base = str(js.location.href).split("/extensions/")[0]
    return pd.read_csv(open_url(f"{base}/files/data/{name}"))


print("Helper load_table() is ready.")

## 2. Reading a CSV file with pandas

The function `pd.read_csv()` reads a CSV file and returns a **DataFrame**.

A DataFrame is a table with:

- rows;
- columns;
- column names;
- values.

We usually store a DataFrame in a variable called `df`.

Here we call our `load_table()` helper, which uses `pd.read_csv()` internally and just adds the location fallbacks described above.

In [ ]:
df = load_table("tensile_lab_raw.csv")

# Show the first rows
# This is usually the first thing we do after reading a file.
df.head()

## 3. First inspection of the data

Before doing calculations, always inspect the dataset.

Basic questions:

- How many rows and columns do we have?
- What are the column names?
- What kind of values are stored in each column?
- Are the values plausible?
- Are there missing values?

This step prevents many mistakes later.

In [ ]:
print("Shape of the table:")
print(df.shape)
print()

print("Column names:")
print(df.columns)
print()

print("Data types:")
print(df.dtypes)

In [ ]:
# A compact summary of the table
# It shows column names, non-missing values, and data types.
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe()

## Exercise 1 — Basic inspection

Use pandas commands to answer these questions:

1. How many rows are in the dataset?
2. How many columns are in the dataset?
3. What are the first 10 rows?
4. What are the last 5 rows?
5. Which columns contain numerical values?

Useful commands:

```python
df.shape
df.head(10)
df.tail()
df.dtypes
```

In [ ]:
# Exercise 1
# Write your commands below.

# 1. Number of rows and columns


# 2. First 10 rows


# 3. Last 5 rows


# 4. Data types

## 4. Selecting columns

A DataFrame usually contains more information than we need at a given moment.

We can select one column:

```python
df["force_N"]
```

or several columns:

```python
df[["sample_id", "force_N", "displacement_mm"]]
```

The double square brackets in the second example are important: we are passing a list of column names.

In [ ]:
# Select one column
force = df["force_N"]

print(type(force))
print(force.head())

In [ ]:
# Select several columns
small_table = df[["sample_id", "treatment", "force_N", "displacement_mm"]]

small_table.head()

## 5. Filtering rows

Very often, we want to select only some rows.

For example:

- only one sample;
- only one treatment;
- only measurements above a certain force;
- only valid measurements.

To filter rows, we write a condition.

Example:

```python
df["sample_id"] == "S01"
```

This gives a column of `True` / `False` values.

Then we use that condition to select rows:

```python
df[df["sample_id"] == "S01"]
```

Remember:

- `=` assigns a value;
- `==` checks equality.

In [ ]:
# Create a Boolean condition
condition = df["sample_id"] == "S01"

condition.head(10)

In [ ]:
# Use the condition to filter rows
sample_s01 = df[df["sample_id"] == "S01"]

sample_s01.head()

In [ ]:
# Filter by treatment
annealed = df[df["treatment"] == "annealed"]

print("Rows in original dataset:", len(df))
print("Rows for annealed samples:", len(annealed))

annealed.head()

### Multiple conditions

To combine conditions in pandas, use:

- `&` for **and**;
- `|` for **or**;
- `~` for **not**.

Each condition should be placed inside parentheses.

Example:

```python
(df["sample_id"] == "S01") & (df["force_N"] > 100)
```

In [ ]:
# Example: rows for sample S01 where force is greater than 100 N
subset = df[(df["sample_id"] == "S01") & (df["force_N"] > 100)]

subset.head()

## Exercise 2 — Filtering

Create three filtered tables:

1. all rows for sample `S03`;
2. all rows where `force_N` is greater than 500;
3. all rows for quenched samples where `displacement_mm` is greater than 0.1.

Then print the number of rows in each filtered table.

In [ ]:
# Exercise 2
# Write your code below.

# 1. Rows for sample S03


# 2. Rows where force_N > 500


# 3. Rows for quenched samples where displacement_mm > 0.1

## 6. Creating calculated columns

The raw dataset contains force and displacement.

For a tensile test, we usually want stress and strain.

We will calculate:

```text
area_mm2 = π × (diameter_mm / 2)^2
stress_MPa = force_N / area_mm2
strain = displacement_mm / gauge_length_mm
```

Important unit note:

```text
1 N / mm² = 1 MPa
```

So if force is in newtons and area is in mm², the stress is in MPa.

In [ ]:
# Create calculated columns

df["area_mm2"] = math.pi * (df["diameter_mm"] / 2) ** 2
df["stress_MPa"] = df["force_N"] / df["area_mm2"]
df["strain"] = df["displacement_mm"] / df["gauge_length_mm"]

# Inspect the new columns
df[["sample_id", "force_N", "displacement_mm", "area_mm2", "stress_MPa", "strain"]].head()

In [ ]:
# Check ranges of the new quantities
print("Stress range [MPa]:")
print(df["stress_MPa"].min(), "to", df["stress_MPa"].max())
print()

print("Strain range [-]:")
print(df["strain"].min(), "to", df["strain"].max())

## Exercise 3 — Add another calculated column

Create a new column called `strain_percent`.

It should be equal to:

```text
strain_percent = strain × 100
```

Then display these columns:

```text
sample_id, strain, strain_percent
```

for the first 10 rows.

In [ ]:
# Exercise 3
# Write your code below.

## 7. Simple data-quality checks

Before saving cleaned data, we should check for obvious problems.

Examples:

- missing values;
- negative forces;
- negative displacements;
- unexpected sample names;
- unexpected treatment labels.

These checks are simple, but they often catch real mistakes.

In [ ]:
# Count missing values in each column
df.isna().sum()

In [ ]:
# Check unique samples and treatments
print("Sample IDs:")
print(df["sample_id"].unique())
print()

print("Treatments:")
print(df["treatment"].unique())

In [ ]:
# Check for negative values in quantities that should not be negative
negative_force_rows = df[df["force_N"] < 0]
negative_displacement_rows = df[df["displacement_mm"] < 0]

print("Rows with negative force:", len(negative_force_rows))
print("Rows with negative displacement:", len(negative_displacement_rows))

### From "raw" to "clean": removing bad rows

The checks above are not just for show — this raw dataset really does contain a few bad measurements:

- some **missing** force readings (`NaN`);
- a few **negative** forces and one negative displacement (sensor glitches).

A negative or missing force makes no physical sense, so we drop those rows to build a **clean** table. We keep the original `df` untouched and create a new table called `df_clean`.

In [ ]:
# Build a clean table by removing physically impossible rows
bad_rows = (
    df["force_N"].isna()
    | (df["force_N"] < 0)
    | (df["displacement_mm"] < 0)
)

df_clean = df[~bad_rows].copy()

print("Rows before cleaning:", len(df))
print("Rows removed:        ", int(bad_rows.sum()))
print("Rows after cleaning: ", len(df_clean))

## Exercise 4 — Sanity checks

Answer these questions with code:

1. How many different samples are in the dataset?
2. How many different treatments are in the dataset?
3. How many rows are present for each sample?
4. What is the maximum force measured for each sample?

Useful methods:

```python
.unique()
.nunique()
.value_counts()
.groupby()
.max()
```

In [ ]:
# Exercise 4
# Write your code below.

# 1. Number of different samples


# 2. Number of different treatments


# 3. Number of rows for each sample


# 4. Maximum force for each sample

## 8. Grouping and summarizing

A common task is to calculate one or more summary values for each sample.

For example:

- maximum stress;
- maximum strain;
- treatment label;
- average diameter.

This turns a long raw table into a compact summary table.

The operation is:

```text
many rows per sample → one summary row per sample
```

In [ ]:
summary_by_sample = (
    df_clean
    .groupby("sample_id")
    .agg(
        treatment=("treatment", "first"),
        diameter_mm=("diameter_mm", "mean"),
        max_force_N=("force_N", "max"),
        max_stress_MPa=("stress_MPa", "max"),
        max_strain=("strain", "max"),
    )
    .reset_index()
)

summary_by_sample

We can also summarize by treatment.

This is useful when we want to compare groups of samples.

In [ ]:
summary_by_treatment = (
    summary_by_sample
    .groupby("treatment")
    .agg(
        n_samples=("sample_id", "count"),
        mean_max_stress_MPa=("max_stress_MPa", "mean"),
        std_max_stress_MPa=("max_stress_MPa", "std"),
        mean_max_strain=("max_strain", "mean"),
    )
    .reset_index()
)

summary_by_treatment

## 9. Saving cleaned data and results

After cleaning and processing data, we often want to save the result.

We will save:

1. the full cleaned table;
2. the summary table by sample;
3. the summary table by treatment.

This is important because other notebooks, scripts, or reports can use the cleaned data without repeating all manual steps.

In [ ]:
clean_file = RESULTS_DIR / "tensile_clean.csv"
summary_sample_file = RESULTS_DIR / "tensile_summary_by_sample.csv"
summary_treatment_file = RESULTS_DIR / "tensile_summary_by_treatment.csv"

# index=False avoids saving the DataFrame index as an extra column
df_clean.to_csv(clean_file, index=False)
summary_by_sample.to_csv(summary_sample_file, index=False)
summary_by_treatment.to_csv(summary_treatment_file, index=False)

print("Saved files:")
print(clean_file)
print(summary_sample_file)
print(summary_treatment_file)

In [ ]:
# Check that the saved file can be read again
check = pd.read_csv(clean_file)

check.head()

## 10. Quick visual check

This lesson is mainly about I/O and tabular data.

However, a quick plot is often useful to check whether the processed data look reasonable.

Here we plot stress versus strain for one sample.

In [ ]:
sample_id = "S01"
sample = df_clean[df_clean["sample_id"] == sample_id]

plt.figure(figsize=(6, 4))
plt.plot(sample["strain"], sample["stress_MPa"], marker="o", markersize=3)
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Stress-strain curve for sample {sample_id}")
plt.grid(True)
plt.show()

## 11. Mini-project — Clean and summarize a dataset

Use what we learned today to build a small analysis workflow.

### Task

Starting from `df`, produce:

1. a cleaned table with calculated columns;
2. a summary table by sample;
3. a summary table by treatment;
4. one simple plot for a chosen sample;
5. saved CSV files in the `results` folder.

### Minimum requirements

Your notebook should contain:

- a title;
- a short description of the dataset;
- data loading;
- data inspection;
- calculated columns;
- basic sanity checks;
- saved outputs.

### Optional extension

Repeat the same type of workflow for `hardness_positions.csv`.

In [ ]:
# Mini-project — starter scaffold
# Fill in each step. You can reuse code from the sections above.

# 1. Load the data (it is already in df / df_clean above; you may also reload it here)


# 2. Make sure the calculated columns exist (area_mm2, stress_MPa, strain)


# 3. Build a summary table by sample (groupby + agg)


# 4. Build a summary table by treatment


# 5. Make one stress-strain plot for a sample of your choice


# 6. Save your summary tables into the results folder with to_csv(...)

## Optional extension — Hardness data

The folder also contains a smaller dataset:

```text
data/hardness_positions.csv
```

This dataset contains repeated hardness measurements at several positions on the same samples.

Columns:

- `sample_id`;
- `treatment`;
- `position_id`;
- `hardness_HV`;
- `is_valid` (some measurements were flagged as invalid).

Possible tasks:

1. read the file;
2. keep only rows where `is_valid == True`;
3. calculate mean and standard deviation of hardness for each sample;
4. calculate mean hardness for each treatment;
5. save a summary CSV.

In [ ]:
# Optional extension
# Read hardness_positions.csv and summarize the valid measurements.

hardness_file = DATA_DIR / "hardness_positions.csv"

# Your code here

## Optional demonstration — Reading metadata from JSON

Not all useful information is stored in CSV files.

Sometimes metadata are stored in JSON files.

JSON is useful for structured information such as:

- instrument name;
- operator;
- sample preparation details;
- project information;
- analysis parameters.

This is optional, but it is good to know that Python can read these files too.

In [ ]:
import json

# Try the data folder first, then next to the notebook (browser-safe).
try:
    f = open(DATA_DIR / "sample_metadata.json", "r")
except OSError:
    f = open(Path("sample_metadata.json"), "r")

with f:
    metadata = json.load(f)

metadata

# End of Lesson 02

## What we learned

Today we learned how to:

- locate a data file;
- read a CSV file with pandas;
- inspect a DataFrame;
- select columns;
- filter rows;
- create calculated columns;
- perform simple sanity checks;
- summarize data by sample and treatment;
- save cleaned data and summary tables;
- make a quick plot to check the result.

## Main message

Reading a file is not the end of the analysis.

A good data workflow is:

```text
read → inspect → calculate → check → summarize → save
```

In the next lesson, we will focus more on plotting and scientific interpretation.